# Perceptron

## What is a Perceptron?

A perceptron is the simplest possible model of a neuron, and it's the building block of every neural network we'll build.

Here's the idea: a perceptron takes some inputs, multiplies each one by a weight (a number that says how much that input matters), adds them all up along with a bias term (a fixed offset), and then asks a single yes-or-no question: is that sum greater than zero? If yes, it outputs 1. If no, it outputs 0.

That makes it a **binary classifier** — it can assign any input to one of two categories.

The bias term works like an intercept in linear regression. Changing the bias shifts the decision boundary without changing its angle. Changing the weights rotates the boundary. Together, weights and bias define a straight line (or a flat plane in higher dimensions) that separates the two classes.

## What a Perceptron Can and Can't Do

A perceptron can only learn decision rules where the boundary between classes is a straight line. This is called **linear separability**. If you can draw a single straight line that correctly separates your two classes, a perceptron can learn it. If you can't, the perceptron will never converge — it will keep adjusting its weights forever without finding a solution.

We'll see exactly what that failure looks like at the end of this notebook.

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import numpy as np

## The Perceptron Class

The implementation below captures the full perceptron in about 20 lines of code.

**Initialization:** The weights (including the bias, stored as the first weight) start at random values.

**Prediction:** For a given input, we compute the weighted sum and apply the threshold: output 1 if the sum is positive, 0 otherwise.

**Training — how it learns:** We pass through the data for a set number of epochs. For each example, we make a prediction. If the prediction is correct, we do nothing. If it's wrong, we nudge each weight in the direction that would have given the right answer. The size of that nudge is controlled by the **learning rate** — a small number that prevents the weights from swinging wildly with each mistake.

The weight update rule is: new weight = old weight + learning_rate * (correct_label - predicted_label) * input. If the model predicted 0 but the answer was 1, the term `(label - prediction)` is positive, so we increase the weights on inputs that were active. If it predicted 1 but should have predicted 0, we decrease them. Over many examples, the weights converge on values that get the boundary right.

In [ ]:
class Perceptron(object):

    def __init__(self, no_of_inputs):
        self.weights = np.random.random(no_of_inputs + 1)

    def predict(self, inputs):
        summation = np.dot(np.insert(inputs, 0, 1), self.weights)
        if summation > 0:
          activation = 1
        else:
          activation = 0
        return activation

    def train(self, training_inputs, labels, epochs = 100, learning_rate=0.01):
        for _ in range(epochs):
            for inputs, label in zip(training_inputs, labels):
                prediction = self.predict(inputs)
                self.weights += learning_rate * (label - prediction) * np.insert(inputs, 0, 1)

    def __str__(self):
        return 'Weights: ' + str(self.weights)

## Setting Up the Training Data

We'll use the four possible two-bit input combinations as our dataset — the standard logical truth table. We'll train the perceptron on two classic logical functions: OR and AND.

- **OR** outputs 1 if at least one input is 1. Three out of four rows are 1.
- **AND** outputs 1 only if both inputs are 1. Only the last row is 1.

Both of these are linearly separable, so the perceptron should be able to learn them perfectly.

In [ ]:
truth_table_inputs = [[0,0], [0,1], [1,0], [1,1]]
or_outputs = [0, 1, 1, 1]
and_outputs = [0, 0, 0, 1]

## Training a Perceptron on OR

We create a new perceptron with random weights, check its predictions before training (expect them to be mostly wrong or arbitrary), then train it and check again. After training, it should correctly output 0 only for [0,0] and 1 for everything else.

In [ ]:
p_or = Perceptron(2)

In [ ]:
print(p_or)

In [ ]:
[p_or.predict(i) for i in truth_table_inputs]

In [ ]:
p_or.train(truth_table_inputs, or_outputs)

In [ ]:
print(p_or)

In [ ]:
p_or.predict([0,0])
print('\n')
p_or.predict([1,0])
print('\n')
p_or.predict([0,1])
print('\n')
p_or.predict([1,1])

## Training a Perceptron on AND

AND is also linearly separable, so the perceptron should learn it just as cleanly. Notice that compared to OR, the only thing that changes is the labels — [1,1] is the only positive case. The perceptron will need to push its decision boundary further up so that only the [1,1] combination clears the threshold.

In [ ]:
p_and = Perceptron(2)

In [ ]:
print(p_and)

In [ ]:
[p_and.predict(i) for i in truth_table_inputs]

In [ ]:
p_and.train(truth_table_inputs, and_outputs)

In [ ]:
print(p_and)

In [ ]:
p_and.predict([0,0])
print('\n')
p_and.predict([1,0])
print('\n')
p_and.predict([0,1])
print('\n')
p_and.predict([1,1])

## OR vs AND: Same Weights, Different Bias

The perceptron for OR uses w1=1, w2=1, b=−0.5. The perceptron for AND uses the exact same weights but a different bias: b=−1.5.

The decision boundary is where w1·x1 + w2·x2 + b = 0, which rearranges to x2 = −x1 − b. Changing only b shifts the intercept without rotating the line — both boundaries have slope −1, but AND's sits higher, requiring both inputs to be 1 before the sum clears the threshold.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
x_line = np.linspace(-0.3, 1.3, 100)
pts = [[0,0],[0,1],[1,0],[1,1]]

for ax, (title, b, labels) in zip(axes, [
    ('OR  (w1=1, w2=1, b=-0.5)',  -0.5, [0,1,1,1]),
    ('AND  (w1=1, w2=1, b=-1.5)', -1.5, [0,0,0,1]),
]):
    boundary = -x_line - b   # decision boundary: w1*x1 + w2*x2 + b = 0  =>  x2 = -x1 - b
    ax.plot(x_line, boundary, 'k--', linewidth=2, label='Decision boundary', zorder=2)
    ax.fill_between(x_line, boundary, 1.6, alpha=0.10, color='steelblue', label='Predicts 1')
    ax.fill_between(x_line, -0.6, boundary, alpha=0.10, color='salmon', label='Predicts 0')
    for (x1, x2), lbl in zip(pts, labels):
        ax.scatter(x1, x2, s=220,
                   c='steelblue' if lbl == 1 else 'salmon',
                   edgecolors='k', zorder=3, linewidths=1.5)
        ax.annotate(f'({x1},{x2})={lbl}', (x1, x2),
                    textcoords='offset points', xytext=(8, 5), fontsize=9)
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.6, 1.6)
    ax.axhline(0, color='gray', linewidth=0.4)
    ax.axvline(0, color='gray', linewidth=0.4)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    ax.set_title(title)
    ax.legend(fontsize=8, loc='upper right')

fig.suptitle('Same weights (w1=1, w2=1) — changing only the bias shifts the line in parallel.',
             fontsize=11)
plt.tight_layout()
plt.show()

## The Limits of a Perceptron: XOR

Now let's try something the perceptron fundamentally cannot do.

XOR ("exclusive or") outputs 1 when the inputs are *different* — [0,1] and [1,0] — and 0 when they're the same — [0,0] and [1,1]. If you plot these four points and try to draw a single straight line that puts the 1s on one side and the 0s on the other, you'll find it's impossible. The 1s are in opposite corners of the grid, and the 0s are in the other two corners. No straight line can separate them.

This is the core limitation of the perceptron: it can only learn **linearly separable** problems. XOR is not linearly separable, so no matter how long we train — even for 5,000 epochs — the perceptron will never get it right. Watch what happens:

In [ ]:
xor_outputs = [0, 1, 1, 0]

In [ ]:
p_xor = Perceptron(2)

In [ ]:
print(p_xor)

In [ ]:
p_xor.train(truth_table_inputs, xor_outputs, epochs = 5000)

In [ ]:
print(p_xor)

In [ ]:
p_xor.predict([0,0])
print('\n')
p_xor.predict([1,0])
print('\n')
p_xor.predict([0,1])
print('\n')
p_xor.predict([1,1])

The predictions are still wrong (or only partially right by luck). The weights just kept shifting back and forth without settling, because there is no configuration of a single straight-line boundary that can solve XOR.

This failure is historically important — it was one of the key findings that slowed neural network research in the 1970s. The fix, as we'll see in the next notebook, is to chain multiple perceptrons together into a network with hidden layers.

## Interactive: Explore the Decision Boundary

Use the sliders below to manually set the two weights and the bias, and see how the decision boundary moves. This gives you an intuition for what learning is actually doing — finding weight and bias values that put the right points on the right side of the line.

- **Weight 1** and **Weight 2** control the angle (slope) of the boundary.
- **Bias** shifts the boundary left/right without changing its angle.
- Try to find settings that correctly separate the OR outputs (0 for [0,0], 1 for everything else).
- Then try to do the same for XOR — you'll see it's genuinely impossible with a single line.

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

truth_table_inputs_arr = np.array([[0,0], [0,1], [1,0], [1,1]])

def plot_boundary(w1, w2, bias, task):
    if task == 'OR':
        labels = [0, 1, 1, 1]
        title = 'OR (0 for [0,0], 1 otherwise)'
    elif task == 'AND':
        labels = [0, 0, 0, 1]
        title = 'AND (1 only for [1,1])'
    else:
        labels = [0, 1, 1, 0]
        title = 'XOR (try to separate these — it cannot be done with one line)'

    colors = ['red' if l == 0 else 'blue' for l in labels]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(truth_table_inputs_arr[:, 0], truth_table_inputs_arr[:, 1],
               c=colors, s=200, zorder=5)
    for (x, y), label in zip(truth_table_inputs_arr, labels):
        ax.annotate(str(label), (x, y), textcoords='offset points',
                    xytext=(10, 5), fontsize=12)

    # Draw the decision boundary: w1*x1 + w2*x2 + bias = 0  =>  x2 = -(w1*x1 + bias) / w2
    x_vals = np.linspace(-0.5, 1.5, 200)
    if abs(w2) > 1e-6:
        y_vals = -(w1 * x_vals + bias) / w2
        ax.plot(x_vals, y_vals, 'k-', label='Decision boundary')
    else:
        if abs(w1) > 1e-6:
            x_line = -bias / w1
            ax.axvline(x=x_line, color='k', label='Decision boundary')

    ax.set_xlim(-0.3, 1.3)
    ax.set_ylim(-0.3, 1.3)
    ax.set_xlabel('Input 1')
    ax.set_ylabel('Input 2')
    ax.set_title(title + f'\nw1={w1:.2f}, w2={w2:.2f}, bias={bias:.2f}')
    ax.legend()
    plt.tight_layout()
    plt.show()

widgets.interact(
    plot_boundary,
    w1=widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1, description='Weight 1'),
    w2=widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1, description='Weight 2'),
    bias=widgets.FloatSlider(value=-0.5, min=-3.0, max=3.0, step=0.1, description='Bias'),
    task=widgets.Dropdown(options=['OR', 'AND', 'XOR'], value='OR', description='Task')
)